librerias y utilities para empezar a hacer los dataframes

In [50]:
import requests
import json

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pathlib import Path
from google.colab import files

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    LongType,
    ArrayType,
    MapType
)

In [30]:
spark = (
    SparkSession.builder
    .appName("RestCountriesETL")
    .getOrCreate()
)

Conexión a la API

In [35]:
from getpass import getpass

API_TOKEN = getpass("REST Countries API token: ")

REST Countries API token: ··········


In [32]:
def Extraer(api_token):
    countries = [
        "estonia",
        "mexico",
        "india",
        "canada",
        "australia"
    ]

    base_url = "https://api.restcountries.com/countries/v5"

    headers = {
        "Authorization": f"Bearer {api_token}"
    }

    raw_countries = []

    for country in countries:
        response = requests.get(
            base_url,
            params={"q": country},
            headers=headers,
            timeout=30
        )

        response.raise_for_status()

        payload = response.json()
        objects = payload["data"]["objects"]

        country_match = next(
            (
                obj for obj in objects
                if obj["names"]["common"].lower() == country.lower()
            ),
            None
        )

        if country_match is None:
            raise ValueError(
                f"No se encontró una coincidencia exacta para {country}"
            )

        raw_countries.append(country_match)

    json_rows = [
        json.dumps(country, ensure_ascii=False)
        for country in raw_countries
    ]

    raw_df = spark.read.json(
        spark.sparkContext.parallelize(json_rows)
    )

    return raw_df

In [36]:
raw_data = Extraer(API_TOKEN)
print(raw_data)

DataFrame[_match: array<struct<path:string,value:string>>, _meta: struct<lastUpdatedTimestamp:bigint>, area: struct<kilometers:bigint,miles:double>, assets: array<string>, borders: array<string>, calling_codes: array<string>, capitals: array<struct<attributes:struct<administrative:boolean,constitutional:boolean,executive:boolean,judicial:boolean,legislative:boolean,primary:boolean>,coordinates:struct<lat:double,lng:double>,name:string>>, cars: struct<driving_side:string,signs:array<string>>, classification: struct<dependency:boolean,dependency_type:string,disputed:boolean,iso_status:string,sovereign:boolean,un_member:boolean,un_observer:boolean>, codes: struct<alpha_2:string,alpha_3:string,ccn3:string,cioc:string,fifa:string,fips:string,gec:string>, continents: array<string>, coordinates: struct<lat:bigint,lng:bigint>, currencies: array<struct<code:string,name:string,symbol:string>>, date: struct<academic_year_start:struct<day:bigint,month:bigint>,fiscal_year_start:struct<corporate:str

In [37]:
raw_df.select(
    F.col("codes.alpha_3").alias("country_code"),
    F.col("names.common").alias("country_name"),
    F.col("capitals")[0]["name"].alias("capital")
).show(truncate=False)

+------------+------------+-----------+
|country_code|country_name|capital    |
+------------+------------+-----------+
|EST         |Estonia     |Tallinn    |
|MEX         |Mexico      |Mexico City|
|IND         |India       |New Delhi  |
|CAN         |Canada      |Ottawa     |
|AUS         |Australia   |Canberra   |
+------------+------------+-----------+



Comienza la transformación de data

In [38]:

def Transformar(raw_df):

    CountryDF = (
        raw_df
        .select(
            F.upper(F.col("codes.alpha_3")).alias("country_code"),
            F.trim(F.col("names.official")).alias("official_name"),
            F.trim(F.col("capitals")[0]["name"]).alias("capital"),
            F.trim(F.col("region")).alias("region"),
            F.trim(F.col("subregion")).alias("subregion"),
            F.col("area.kilometers").cast("double").alias("area_km2"),
            F.col("population").cast("long").alias("population")
        )
        .filter(F.col("country_code").isNotNull())
        .dropDuplicates(["country_code"])
    )


    language_source = (
        raw_df
        .select(
            F.upper(F.col("codes.alpha_3")).alias("country_code"),
            F.explode_outer(F.col("languages")).alias("language")
        )
    )

    LanguageDF = (
        language_source
        .select(
            F.lower(F.col("language.iso639_3")).alias("language_code"),
            F.trim(F.col("language.name")).alias("language_name"),
            F.trim(F.col("language.native_name")).alias("native_name")
        )
        .filter(F.col("language_code").isNotNull())
        .dropDuplicates(["language_code"])
    )

    CountryLanguageDF = (
        language_source
        .select(
            F.col("country_code"),
            F.lower(F.col("language.iso639_3")).alias("language_code")
        )
        .filter(
            F.col("country_code").isNotNull()
            & F.col("language_code").isNotNull()
        )
        .dropDuplicates(["country_code", "language_code"])
    )


    currency_source = (
        raw_df
        .select(
            F.upper(F.col("codes.alpha_3")).alias("country_code"),
            F.explode_outer(F.col("currencies")).alias("currency")
        )
    )

    CurrencyDF = (
        currency_source
        .select(
            F.upper(F.col("currency.code")).alias("currency_code"),
            F.trim(F.col("currency.name")).alias("currency_name"),
            F.trim(F.col("currency.symbol")).alias("symbol")
        )
        .filter(F.col("currency_code").isNotNull())
        .dropDuplicates(["currency_code"])
    )

    CountryCurrencyDF = (
        currency_source
        .select(
            F.col("country_code"),
            F.upper(F.col("currency.code")).alias("currency_code")
        )
        .filter(
            F.col("country_code").isNotNull()
            & F.col("currency_code").isNotNull()
        )
        .dropDuplicates(["country_code", "currency_code"])
    )


    BorderDF = (
        raw_df
        .select(
            F.upper(F.col("codes.alpha_3")).alias("country_code"),
            F.explode_outer(F.col("borders")).alias("border_country_code")
        )
        .filter(
            F.col("country_code").isNotNull()
            & F.col("border_country_code").isNotNull()
        )
        .withColumn(
            "border_country_code",
            F.upper(F.trim(F.col("border_country_code")))
        )
        .dropDuplicates(["country_code", "border_country_code"])
    )


    return {
        "CountryDF": CountryDF,
        "LanguageDF": LanguageDF,
        "CountryLanguageDF": CountryLanguageDF,
        "CurrencyDF": CurrencyDF,
        "CountryCurrencyDF": CountryCurrencyDF,
        "BorderDF": BorderDF
    }

In [39]:
tables = Transformar(raw_df)

In [40]:
tables["CountryDF"].show(truncate=False)

+------------+-------------------------+-----------+--------+-------------------------+---------+----------+
|country_code|official_name            |capital    |region  |subregion                |area_km2 |population|
+------------+-------------------------+-----------+--------+-------------------------+---------+----------+
|AUS         |Commonwealth of Australia|Canberra   |Oceania |Australia and New Zealand|7692024.0|27921150  |
|CAN         |Canada                   |Ottawa     |Americas|North America            |9984670.0|41798407  |
|EST         |Republic of Estonia      |Tallinn    |Europe  |Northern Europe          |45227.0  |1360745   |
|IND         |Republic of India        |New Delhi  |Asia    |Southern Asia            |3287590.0|1429404000|
|MEX         |United Mexican States    |Mexico City|Americas|North America            |1964375.0|131231919 |
+------------+-------------------------+-----------+--------+-------------------------+---------+----------+



In [41]:
tables["LanguageDF"].show(truncate=False)

+-------------+-------------+-----------+
|language_code|language_name|native_name|
+-------------+-------------+-----------+
|eng          |English      |English    |
|est          |Estonian     |eesti      |
|fra          |French       |français   |
|hin          |Hindi        |हिन्दी     |
|spa          |Spanish      |español    |
|tam          |Tamil        |தமிழ்      |
+-------------+-------------+-----------+



In [42]:
tables["CountryLanguageDF"].show(truncate=False)

+------------+-------------+
|country_code|language_code|
+------------+-------------+
|EST         |est          |
|MEX         |spa          |
|CAN         |eng          |
|IND         |tam          |
|IND         |eng          |
|CAN         |fra          |
|IND         |hin          |
|AUS         |eng          |
+------------+-------------+



In [43]:
tables["CurrencyDF"].show(truncate=False)

+-------------+-----------------+------+
|currency_code|currency_name    |symbol|
+-------------+-----------------+------+
|AUD          |Australian dollar|$     |
|CAD          |Canadian dollar  |$     |
|EUR          |Euro             |€     |
|INR          |Indian rupee     |₹     |
|MXN          |Mexican peso     |$     |
+-------------+-----------------+------+



In [44]:
tables["BorderDF"].show(truncate=False)

+------------+-------------------+
|country_code|border_country_code|
+------------+-------------------+
|EST         |RUS                |
|EST         |LVA                |
|MEX         |USA                |
|MEX         |GTM                |
|MEX         |BLZ                |
|IND         |BGD                |
|IND         |PAK                |
|IND         |BTN                |
|CAN         |USA                |
|IND         |NPL                |
|IND         |CHN                |
|IND         |MMR                |
+------------+-------------------+



Load del ETL para descargar todo en un ZIP de los CSV

In [45]:


def Cargar(tables):
    base_path = Path("data/processed/tables")
    base_path.mkdir(parents=True, exist_ok=True)

    for table_name, df in tables.items():
        output_path = str(base_path / table_name)

        (
            df
            .coalesce(1)
            .write
            .mode("overwrite")
            .option("header", True)
            .csv(output_path)
        )

In [47]:
Cargar(tables)

In [51]:
!zip -r /content/processed_tables.zip /content/data/processed/tables
files.download("/content/processed_tables.zip")

updating: content/data/processed/tables/ (stored 0%)
updating: content/data/processed/tables/CurrencyDF/ (stored 0%)
updating: content/data/processed/tables/CurrencyDF/_SUCCESS (stored 0%)
updating: content/data/processed/tables/CurrencyDF/._SUCCESS.crc (stored 0%)
updating: content/data/processed/tables/CurrencyDF/part-00000-2d4640bd-fc4f-4f0e-bc2b-08fb0c3e220f-c000.csv (deflated 16%)
updating: content/data/processed/tables/CurrencyDF/.part-00000-2d4640bd-fc4f-4f0e-bc2b-08fb0c3e220f-c000.csv.crc (stored 0%)
updating: content/data/processed/tables/CountryCurrencyDF/ (stored 0%)
updating: content/data/processed/tables/CountryCurrencyDF/_SUCCESS (stored 0%)
updating: content/data/processed/tables/CountryCurrencyDF/._SUCCESS.crc (stored 0%)
updating: content/data/processed/tables/CountryCurrencyDF/.part-00000-959ef06f-ebb6-4d47-be26-7fe247f4998e-c000.csv.crc (stored 0%)
updating: content/data/processed/tables/CountryCurrencyDF/part-00000-959ef06f-ebb6-4d47-be26-7fe247f4998e-c000.csv (defl

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Comprobaciones:
• Idiomas por país
• Monedas usadas por países de una región
• Lista de países fronterizos por país
• Ranking de países por densidad poblacional (population/área)

In [52]:
languages_by_country = (
    tables["CountryDF"]
    .join(
        tables["CountryLanguageDF"],
        on="country_code",
        how="left"
    )
    .join(
        tables["LanguageDF"],
        on="language_code",
        how="left"
    )
    .groupBy(
        "country_code",
        "official_name"
    )
    .agg(
        F.collect_set("language_name").alias("languages")
    )
    .orderBy("official_name")
)

languages_by_country.show(truncate=False)

+------------+-------------------------+-----------------------+
|country_code|official_name            |languages              |
+------------+-------------------------+-----------------------+
|CAN         |Canada                   |[French, English]      |
|AUS         |Commonwealth of Australia|[English]              |
|EST         |Republic of Estonia      |[Estonian]             |
|IND         |Republic of India        |[Hindi, English, Tamil]|
|MEX         |United Mexican States    |[Spanish]              |
+------------+-------------------------+-----------------------+



In [53]:
currencies_by_region = (
    tables["CountryDF"]
    .join(
        tables["CountryCurrencyDF"],
        on="country_code",
        how="left"
    )
    .join(
        tables["CurrencyDF"],
        on="currency_code",
        how="left"
    )
    .select(
        "region",
        "country_code",
        "official_name",
        "currency_code",
        "currency_name"
    )
    .orderBy(
        "region",
        "official_name"
    )
)

currencies_by_region.show(truncate=False)

+--------+------------+-------------------------+-------------+-----------------+
|region  |country_code|official_name            |currency_code|currency_name    |
+--------+------------+-------------------------+-------------+-----------------+
|Americas|CAN         |Canada                   |CAD          |Canadian dollar  |
|Americas|MEX         |United Mexican States    |MXN          |Mexican peso     |
|Asia    |IND         |Republic of India        |INR          |Indian rupee     |
|Europe  |EST         |Republic of Estonia      |EUR          |Euro             |
|Oceania |AUS         |Commonwealth of Australia|AUD          |Australian dollar|
+--------+------------+-------------------------+-------------+-----------------+



In [55]:
borders_by_country = (
    tables["CountryDF"]
    .join(
        tables["BorderDF"],
        on="country_code",
        how="left"
    )
    .groupBy(
        "country_code",
        "official_name"
    )
    .agg(
        F.collect_set("border_country_code").alias("border_countries")
    )
    .orderBy("official_name")
)

borders_by_country.show(truncate=False)

+------------+-------------------------+------------------------------+
|country_code|official_name            |border_countries              |
+------------+-------------------------+------------------------------+
|CAN         |Canada                   |[USA]                         |
|AUS         |Commonwealth of Australia|[]                            |
|EST         |Republic of Estonia      |[LVA, RUS]                    |
|IND         |Republic of India        |[BTN, PAK, MMR, NPL, BGD, CHN]|
|MEX         |United Mexican States    |[GTM, BLZ, USA]               |
+------------+-------------------------+------------------------------+



In [57]:
from pyspark.sql.window import Window

ranking_window = Window.orderBy(
    F.desc("population_density")
)

population_density_ranking = (
    population_density_ranking
    .withColumn(
        "rank",
        F.row_number().over(ranking_window)
    )
    .select(
        "rank",
        "country_code",
        "official_name",
        "population",
        "area_km2",
        "population_density"
    )
)

population_density_ranking.show(truncate=False)

+----+------------+-------------------------+----------+---------+------------------+
|rank|country_code|official_name            |population|area_km2 |population_density|
+----+------------+-------------------------+----------+---------+------------------+
|1   |IND         |Republic of India        |1429404000|3287590.0|434.79            |
|2   |MEX         |United Mexican States    |131231919 |1964375.0|66.81             |
|3   |EST         |Republic of Estonia      |1360745   |45227.0  |30.09             |
|4   |CAN         |Canada                   |41798407  |9984670.0|4.19              |
|5   |AUS         |Commonwealth of Australia|27921150  |7692024.0|3.63              |
+----+------------+-------------------------+----------+---------+------------------+

